# Gold 1: chunk the parsed filings

Task `chunk_structured` / `chunk_fixed` of the job `indexing`. Reads the parser's elements from `silver_parsed`
(not the Markdown of `silver_convert`) and chunks every filing whose parts are **all** parsed and that is not
chunked yet for this `strategy`. The logic is plain Python in this folder, tested in `tests/test_gold.py`:

| Module | Does |
| --- | --- |
| [elements.py](./elements.py) | Drops the noise (page headers, footers, numbers, figures), finds PART / Item headings, reads company, period and form from the file name |
| [tables.py](./tables.py) | Table HTML to rows to text, and row groups that repeat the header row |
| [sections.py](./sections.py) | Elements to sections (by heading) and tables (never split, with their units line, merged across a page break) |
| [text_split.py](./text_split.py) | LangChain `RecursiveCharacterTextSplitter`, sizes in tiktoken tokens |
| [chunking.py](./chunking.py) | The strategies: `structured` (parents and children) and `fixed` (the baseline) |

| Strategy | `gold_children` (searched) | `gold_parents` (read by the LLM) |
| --- | --- | --- |
| `structured` | Pieces of about `child_tokens` of a parent; row groups of a table | Sections up to `parent_tokens`; whole tables |
| `fixed` | The text cut every `fixed_tokens`, with overlap | none: a chunk is its own context |

**In parallel.** One filing is one group of `applyInPandas`, so the filings are chunked on the workers at the
same time. The gold modules are shipped with the function (cloudpickle by value), LangChain and tiktoken are
installed by `%pip` (on the workers too), and tiktoken's vocabulary is read from a volume that the driver fills
once, because workers may have no internet access. If the workers cannot run it, set `parallel` to `false`: the
same function then runs on the driver.

**Save points.** Filings are chunked `pdfs_per_batch` at a time, and each batch is saved before the next.
Serverless compute has no `cache()`, so a batch is written once to a staging table and both gold tables are
filled from it. Rows are inserted by key, so a rerun after a crash adds nothing twice.

**To chunk again** (new settings): `DELETE FROM gold_children WHERE strategy = '<strategy>'` (and, for
structured, `DELETE FROM gold_parents`), then run. The embeddings of the deleted rows are lost and paid again.

In [ ]:
%pip install -q langchain-text-splitters==1.1.3 tiktoken==0.14.0

In [ ]:
import sys

sys.path.insert(0, "../common")
import pandas as pd
from pyspark import cloudpickle
from pyspark.sql import functions as F

import chunking
import elements
import sections
import tables
import text_split
from chunking import STRATEGIES, Settings
from gold_tables import create_gold_tables
from names import Names, insert_new

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")
dbutils.widgets.text("gold_schema", "finance_bench_gold")
dbutils.widgets.dropdown("strategy", "structured", list(STRATEGIES))
dbutils.widgets.text("parent_tokens", "900")
dbutils.widgets.text("child_tokens", "200")
dbutils.widgets.text("child_overlap", "30")
dbutils.widgets.text("fixed_tokens", "600")
dbutils.widgets.text("fixed_overlap", "60")
dbutils.widgets.text("pdfs_per_batch", "50")
dbutils.widgets.dropdown("parallel", "true", ["true", "false"])

w = dbutils.widgets.get
n = Names(w("catalog"), w("bronze_schema"), w("silver_schema"), w("gold_schema"))
strategy = w("strategy")
settings = Settings(parent_tokens=int(w("parent_tokens")), child_tokens=int(w("child_tokens")),
                    child_overlap=int(w("child_overlap")), fixed_tokens=int(w("fixed_tokens")),
                    fixed_overlap=int(w("fixed_overlap")))
pdfs_per_batch = int(w("pdfs_per_batch"))
parallel = w("parallel") == "true"
staging = f"{n.catalog}.{n.gold_schema}._chunks_staging_{strategy}"  # one per strategy: the two tasks run at once

create_gold_tables(spark, n)

# tiktoken's vocabulary: downloaded once by the driver into the volume, read from there by every worker.
import tiktoken  # noqa: E402

text_split.use_cache(n.tiktoken_cache)
tiktoken.get_encoding(text_split.ENCODING)

# The gold modules go with the function to the workers, by value: nothing needs to be importable there.
for module in (elements, tables, sections, text_split, chunking):
    cloudpickle.register_pickle_by_value(module)

In [ ]:
# The rows the function returns: children and parents in one schema, told apart by `level`.
OUT_COLUMNS = ["level", "chunk_id", "strategy", "parent_id", "chunk_no", "parent_no", "path", "company",
               "fiscal_period", "doc_type", "chunk_type", "heading_path", "page_start", "page_end", "text",
               "embed_text", "parent_text", "n_tokens"]
OUT_SCHEMA = """level string, chunk_id string, strategy string, parent_id string, chunk_no int, parent_no int,
  path string, company string, fiscal_period string, doc_type string, chunk_type string, heading_path string,
  page_start int, page_end int, text string, embed_text string, parent_text string, n_tokens int"""
cache_dir = n.tiktoken_cache


def chunk_pdf(pdf: pd.DataFrame) -> pd.DataFrame:
    """One filing's elements -> its children and parents. Runs on a worker (or on the driver, parallel=false)."""
    text_split.use_cache(cache_dir)
    pdf = pdf.sort_values(["part_no", "pos"])
    els = [{"page_no": int(r.page_no), "type": r.type, "content": r.content} for r in pdf.itertuples()]
    parents, children = chunking.chunk(pdf["path"].iloc[0], els, strategy, settings)
    rows = [{"level": "child", **c} for c in children] + [{"level": "parent", **p} for p in parents]
    return pd.DataFrame(rows, columns=OUT_COLUMNS)


def to_chunk():
    """Filings whose parts are all parsed and that have no chunks of this strategy yet."""
    return [r.path for r in spark.sql(f"""
    SELECT p.path
    FROM (SELECT path, count(*) AS n FROM {n.parts} GROUP BY path) t
    JOIN (SELECT path, count(*) AS n FROM {n.parsed} GROUP BY path) p ON p.path = t.path AND p.n = t.n
    WHERE p.path NOT IN (SELECT DISTINCT path FROM {n.children} WHERE strategy = '{strategy}')
    ORDER BY p.path""").collect()]


def elements_of(paths):
    """The parser's elements of these filings, one row each, with the real PDF page (as silver_convert counts it)."""
    return spark.sql(f"""
    SELECT d.path, d.part_no, e.pos,
           d.page_offset + coalesce(e.value:bbox[0].page_id::INT, 0) + 1 AS page_no,
           e.value:type::STRING AS type, e.value:content::STRING AS content
    FROM {n.parsed} d, LATERAL variant_explode(d.parsed:document:elements) e
    """).filter(F.col("path").isin(paths))

In [ ]:
def chunk_batch(paths):
    """Chunk a batch of filings and save it. Returns (children, parents) stored."""
    els = elements_of(paths)
    if parallel:
        out = els.groupBy("path").applyInPandas(chunk_pdf, OUT_SCHEMA)
    else:
        local = els.toPandas()
        out = spark.createDataFrame(pd.concat([chunk_pdf(g) for _, g in local.groupby("path")] or
                                              [pd.DataFrame(columns=OUT_COLUMNS)]), OUT_SCHEMA)
    out.write.mode("overwrite").saveAsTable(staging)  # chunk once, write twice (serverless has no cache())

    done = F.current_timestamp().alias("chunked_at")
    staged = spark.table(staging)
    kids = staged.filter("level = 'child'").select(
        "chunk_id", "strategy", "parent_id", "chunk_no", "path", "company", "fiscal_period", "doc_type", "chunk_type",
        "heading_path", "page_start", "page_end", "text", "embed_text", "parent_text", "n_tokens",
        F.lit(None).cast("array<float>").alias("embedding"), done)
    pars = staged.filter("level = 'parent'").select(
        "parent_id", "parent_no", "path", "company", "fiscal_period", "doc_type", "chunk_type", "heading_path",
        "page_start", "page_end", "text", "n_tokens", done)
    insert_new(spark, pars, n.parents, ["parent_id"])  # parents first: a child never points to a missing parent
    insert_new(spark, kids, n.children, ["chunk_id"])
    return kids.count(), pars.count()


todo = to_chunk()
print(f"{len(todo)} filings to chunk with strategy '{strategy}' (parallel={parallel})")
for i in range(0, len(todo), pdfs_per_batch):
    batch = todo[i:i + pdfs_per_batch]
    kids, pars = chunk_batch(batch)
    print(f"filings {i + 1}-{i + len(batch)}: {kids} children, {pars} parents")
spark.sql(f"DROP TABLE IF EXISTS {staging}")

In [ ]:
display(spark.sql(f"""
SELECT strategy, chunk_type, count(DISTINCT path) AS filings, count(*) AS chunks,
       round(avg(n_tokens)) AS avg_tokens, max(n_tokens) AS max_tokens
FROM {n.children} GROUP BY strategy, chunk_type ORDER BY strategy, chunk_type"""))